# Notebook 3: Training a Phi0-Only Potential Model

This notebook demonstrates how to train the Stage 1 ODE potential model on a small
subset of real data. The model learns a potential landscape phi_0(z) whose negative
gradient defines the developmental drift:

$$dz = R_e \cdot (-\beta \nabla \phi_0(z)) \, dt + \sqrt{2D} \, dW$$

where $R_e$ is a per-embryo rate factor inferred in closed form from each trajectory's
context window.

## What to specify at the top
- Which experiments to load
- Number of PCA components
- Training hyperparameters

In [ ]:
import sys, os
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path

REPO_ROOT = "/net/trapnell/vol1/home/nlammers/projects/repositories/morphseq"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "white"})

---
## Configuration

Edit this cell to change experiments, model architecture, and training settings.

In [ ]:
# ===== DATA =====
BUILD_DIR = "/net/trapnell/vol1/home/mdcolon/proj/morphseq/morphseq_playground/metadata/build06_output"
EXPERIMENT_IDS = ["20251207_pbx"]   # <-- change this to use different experiments
N_COMPONENTS = 10
SCALE = True

# ===== MODEL =====
HIDDEN_DIM = 64      # MLP hidden layer width
N_HIDDEN = 2         # number of hidden layers
ACTIVATION = "softplus"  # smooth activation (softplus or elu)
ALPHA_0 = 0.01       # Hessian smoothness penalty weight

# ===== TRAINING =====
N_EPOCHS = 30        # keep small for a quick demo
EPOCH_LENGTH = 500   # samples per epoch
BATCH_SIZE = 32
LR = 1e-3
EVAL_EVERY = 5       # evaluate every N epochs
EVAL_FRACTION = 0.15 # fraction held out for eval

# ===== SAMPLER =====
MIN_CONTEXT = 3
HORIZONS = (1, 2, 3, 4)
GAMMA = 0.5          # class-balanced sampling strength
N_TARGETS = 1        # teacher-forced targets per fragment

DEVICE = "cpu"       # change to "cuda" if GPU available

---
## 1. Load data and split into train/eval

In [ ]:
from dev.dynamo.data import load_trajectories, FragmentDataset, fragment_collate_fn
from dev.dynamo.data.dataset import worker_init_fn
from dev.dynamo.training.trainer import _split_trajectories

dataset = load_trajectories(
    experiment_ids=EXPERIMENT_IDS,
    build_dir=BUILD_DIR,
    n_components=N_COMPONENTS,
    scale=SCALE,
    verbose=True,
)
print(f"\nLoaded {len(dataset)} trajectories, {dataset.n_components} PCs")
print(f"Classes: {dataset.class_names}")

In [ ]:
# Stratified train/eval split
train_ds, eval_ds = _split_trajectories(dataset, eval_fraction=EVAL_FRACTION)
print(f"Train: {len(train_ds)} embryos")
print(f"Eval:  {len(eval_ds)} embryos")

# Fragment datasets
train_frag = FragmentDataset(
    train_ds, min_context=MIN_CONTEXT, horizons=HORIZONS,
    epoch_length=EPOCH_LENGTH, gamma=GAMMA, n_targets=N_TARGETS,
)
eval_frag = FragmentDataset(
    eval_ds, min_context=MIN_CONTEXT, horizons=HORIZONS,
    gamma=0.0, n_targets=1,  # no rebalancing for eval
)

---
## 2. Create model and optimizer

In [ ]:
from dev.dynamo.models.dynamics import Phi0OnlyModel

model = Phi0OnlyModel(
    input_dim=dataset.n_components,
    hidden_dim=HIDDEN_DIM,
    n_hidden=N_HIDDEN,
    activation=ACTIVATION,
    alpha_0=ALPHA_0,
    hessian_n_points=64,
    n_forward_samples=50,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")
print(f"Initial beta = {model.beta.item():.4f}")
print(f"Initial D    = {model.D.item():.6f}")

optimizer = torch.optim.Adam(model.parameters(), lr=LR)

---
## 3. Training loop

In [ ]:
from torch.utils.data import DataLoader
from dev.dynamo.eval import run_evaluation

train_loader = DataLoader(
    train_frag, batch_size=BATCH_SIZE,
    collate_fn=fragment_collate_fn,
    worker_init_fn=worker_init_fn,
    num_workers=0,
)

# Training history
history = {
    "epoch": [], "loss": [], "nll": [], "hessian_penalty": [],
    "beta": [], "D": [], "mean_R_e": [],
    "eval_nll": [], "eval_mse": [], "eval_cal": [], "eval_epoch": [],
}

device = torch.device(DEVICE)

for epoch in range(N_EPOCHS):
    model.train()
    epoch_loss = 0.0
    epoch_nll = 0.0
    epoch_r0 = 0.0
    epoch_R_e = 0.0
    n_batches = 0
    
    for batch in train_loader:
        # Move batch to device
        if DEVICE != "cpu":
            from dev.dynamo.training.trainer import _batch_to_device
            batch = _batch_to_device(batch, device)
        
        result = model(batch)
        loss = result["loss"]
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        optimizer.step()
        
        epoch_loss += loss.item()
        epoch_nll += result["nll"].mean().item()
        epoch_r0 += result["hessian_penalty"].item()
        epoch_R_e += result["R_e"].mean().item()
        n_batches += 1
    
    # Record training metrics
    history["epoch"].append(epoch + 1)
    history["loss"].append(epoch_loss / n_batches)
    history["nll"].append(epoch_nll / n_batches)
    history["hessian_penalty"].append(epoch_r0 / n_batches)
    history["beta"].append(model.beta.item())
    history["D"].append(model.D.item())
    history["mean_R_e"].append(epoch_R_e / n_batches)
    
    # Print progress
    print(f"Epoch {epoch+1:3d}/{N_EPOCHS} | "
          f"loss={history['loss'][-1]:.4f} | "
          f"nll={history['nll'][-1]:.4f} | "
          f"R0={history['hessian_penalty'][-1]:.4f} | "
          f"beta={model.beta.item():.4f} | "
          f"D={model.D.item():.6f} | "
          f"R_e={history['mean_R_e'][-1]:.4f}")
    
    # Evaluation
    if (epoch + 1) % EVAL_EVERY == 0:
        model.eval()
        eval_result = run_evaluation(
            model, eval_frag,
            n_batches=20, batch_size=BATCH_SIZE,
        )
        history["eval_epoch"].append(epoch + 1)
        history["eval_nll"].append(eval_result.metrics["nll"])
        history["eval_mse"].append(eval_result.metrics["mse"])
        history["eval_cal"].append(eval_result.calibration)
        print(f"  >> Eval: NLL={eval_result.metrics['nll']:.4f} | "
              f"MSE={eval_result.metrics['mse']:.6f} | "
              f"Cal={eval_result.calibration:.3f}")
        model.train()

print("\nTraining complete.")

---
## 4. Training curves

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Loss
axes[0, 0].plot(history["epoch"], history["loss"], "b-", lw=1.5)
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Total loss")
axes[0, 0].set_title("Training loss (NLL + α₀·R₀)")

# NLL
axes[0, 1].plot(history["epoch"], history["nll"], "b-", lw=1.5, label="train")
if history["eval_epoch"]:
    axes[0, 1].plot(history["eval_epoch"], history["eval_nll"],
                    "ro-", lw=1.5, markersize=5, label="eval")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("NLL")
axes[0, 1].set_title("Negative log-likelihood")
axes[0, 1].legend()

# Hessian penalty
axes[0, 2].plot(history["epoch"], history["hessian_penalty"], "g-", lw=1.5)
axes[0, 2].set_xlabel("Epoch")
axes[0, 2].set_ylabel("R₀")
axes[0, 2].set_title("Hessian smoothness penalty")

# Beta
axes[1, 0].plot(history["epoch"], history["beta"], "m-", lw=1.5)
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("β")
axes[1, 0].set_title("Drift scale β")

# D
axes[1, 1].plot(history["epoch"], history["D"], "c-", lw=1.5)
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("D")
axes[1, 1].set_title("Diffusion coefficient D")

# Mean R_e
axes[1, 2].plot(history["epoch"], history["mean_R_e"], "r-", lw=1.5)
axes[1, 2].set_xlabel("Epoch")
axes[1, 2].set_ylabel("Mean R_e")
axes[1, 2].set_title("Mean per-embryo rate R_e")

plt.suptitle("Training curves", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Eval metrics over training
if history["eval_epoch"]:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    axes[0].plot(history["eval_epoch"], history["eval_nll"], "ro-", lw=2)
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Eval NLL")
    axes[0].set_title("Eval NLL")
    
    axes[1].plot(history["eval_epoch"], history["eval_mse"], "bo-", lw=2)
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Eval MSE")
    axes[1].set_title("Eval MSE")
    
    axes[2].plot(history["eval_epoch"], history["eval_cal"], "go-", lw=2)
    axes[2].axhline(0.9, color="gray", ls="--", alpha=0.5, label="target=0.90")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Calibration")
    axes[2].set_title("Calibration (90% level)")
    axes[2].legend()
    axes[2].set_ylim(0, 1.05)
    
    plt.suptitle("Evaluation metrics over training", fontsize=13)
    plt.tight_layout()
    plt.show()

---
## 5. Inspect the learned potential landscape

Evaluate phi_0(z) on a grid in PC1-PC2 space and visualize the potential + drift field.

In [ ]:
model.eval()

# Collect all trajectory points for bounds
all_points = np.concatenate([t.trajectory for t in dataset.trajectories], axis=0)
pc1_range = all_points[:, 0]
pc2_range = all_points[:, 1]

# Grid in PC1-PC2 space (other PCs set to 0)
margin = 0.3
n_grid = 40
x_min, x_max = pc1_range.min() - margin, pc1_range.max() + margin
y_min, y_max = pc2_range.min() - margin, pc2_range.max() + margin

xx = np.linspace(x_min, x_max, n_grid)
yy = np.linspace(y_min, y_max, n_grid)
XX, YY = np.meshgrid(xx, yy)

# Build grid points (all PCs beyond 1-2 set to dataset mean)
d = dataset.n_components
mean_pc = all_points.mean(axis=0)
grid_points = np.tile(mean_pc, (n_grid * n_grid, 1))
grid_points[:, 0] = XX.ravel()
grid_points[:, 1] = YY.ravel()

z_grid = torch.from_numpy(grid_points).float().to(DEVICE)

with torch.no_grad():
    phi_vals = model.phi0(z_grid).cpu().numpy().reshape(n_grid, n_grid)

# Compute drift vectors on a coarser grid
n_arrow = 15
xx_a = np.linspace(x_min, x_max, n_arrow)
yy_a = np.linspace(y_min, y_max, n_arrow)
XX_a, YY_a = np.meshgrid(xx_a, yy_a)

arrow_points = np.tile(mean_pc, (n_arrow * n_arrow, 1))
arrow_points[:, 0] = XX_a.ravel()
arrow_points[:, 1] = YY_a.ravel()
z_arrow = torch.from_numpy(arrow_points).float().to(DEVICE)

# gradient needs requires_grad
drift = model.compute_drift_direction(z_arrow).cpu().numpy()
drift = drift.reshape(n_arrow, n_arrow, d)

print(f"Potential range: [{phi_vals.min():.3f}, {phi_vals.max():.3f}]")
print(f"Drift magnitude range: [{np.linalg.norm(drift[..., :2], axis=-1).min():.6f}, "
      f"{np.linalg.norm(drift[..., :2], axis=-1).max():.6f}]")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Panel 1: Potential landscape
ax = axes[0]
cs = ax.contourf(XX, YY, phi_vals, levels=30, cmap="RdYlBu_r")
plt.colorbar(cs, ax=ax, label="φ₀(z)")
# Overlay trajectories
for t in dataset.trajectories[:50]:
    ax.plot(t.trajectory[:, 0], t.trajectory[:, 1], "k-", alpha=0.15, lw=0.5)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Learned potential φ₀(z)")

# Panel 2: Drift field
ax = axes[1]
ax.contourf(XX, YY, phi_vals, levels=30, cmap="RdYlBu_r", alpha=0.3)
# Normalize arrows for visibility
U = drift[:, :, 0]
V = drift[:, :, 1]
magnitude = np.sqrt(U**2 + V**2)
scale = np.percentile(magnitude[magnitude > 0], 90) if (magnitude > 0).any() else 1.0
ax.quiver(XX_a, YY_a, U/scale, V/scale, magnitude,
          cmap="viridis", scale=25, width=0.004, alpha=0.8)
for t in dataset.trajectories[:50]:
    ax.plot(t.trajectory[:, 0], t.trajectory[:, 1], "k-", alpha=0.15, lw=0.5)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Drift field -β∇φ₀(z)")

# Panel 3: Drift magnitude heatmap
ax = axes[2]
# Compute on fine grid
with torch.no_grad():
    drift_fine = model.compute_drift_direction(z_grid).cpu().numpy()
drift_mag = np.linalg.norm(drift_fine[:, :2], axis=1).reshape(n_grid, n_grid)
cs2 = ax.contourf(XX, YY, drift_mag, levels=30, cmap="magma")
plt.colorbar(cs2, ax=ax, label="||drift||")
for t in dataset.trajectories[:50]:
    ax.plot(t.trajectory[:, 0], t.trajectory[:, 1], "w-", alpha=0.2, lw=0.5)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Drift magnitude ||−β∇φ₀||")

plt.suptitle("Learned potential landscape (PC1 vs PC2 slice)", fontsize=14)
plt.tight_layout()
plt.show()

---
## 6. Visualize predictions

Show the model's predictions on eval samples: predicted mean, uncertainty, and forward samples.

In [ ]:
model.eval()

# Get a batch of eval predictions
eval_frag._epoch_length = 100
eval_loader = DataLoader(eval_frag, batch_size=100, collate_fn=fragment_collate_fn)
eval_batch = next(iter(eval_loader))

if DEVICE != "cpu":
    from dev.dynamo.training.trainer import _batch_to_device
    eval_batch = _batch_to_device(eval_batch, device)

pred_result = model.predict(eval_batch)

# Pick samples to show
rng = np.random.default_rng(42)
show_idx = rng.choice(eval_batch.context.shape[0], size=6, replace=False)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for panel, idx in enumerate(show_idx):
    ax = axes[panel]
    L = eval_batch.context_mask[idx].sum().item()
    ctx = eval_batch.context[idx, :L].cpu().numpy()
    tgt = eval_batch.target[idx].cpu().numpy()
    pred_mean = pred_result.predicted_mean[idx].cpu().numpy()
    pred_cov = pred_result.predicted_cov_diag[idx].cpu().numpy()
    
    # Parent trajectory
    traj_idx = eval_batch.embryo_idx[idx].item()
    parent = dataset.trajectories[traj_idx]
    
    pc_x, pc_y = 0, 1
    
    # Parent (gray)
    ax.plot(parent.trajectory[:, pc_x], parent.trajectory[:, pc_y],
            color="lightgray", lw=1, zorder=1)
    
    # Context (blue)
    ax.plot(ctx[:, pc_x], ctx[:, pc_y], "b-", lw=2.5, zorder=3)
    ax.scatter(ctx[-1, pc_x], ctx[-1, pc_y], color="blue", s=50,
               zorder=4, marker=">", edgecolors="k", linewidths=0.5)
    
    # Forward samples (light blue scatter)
    if pred_result.forward_samples is not None:
        samples = pred_result.forward_samples[idx].cpu().numpy()
        ax.scatter(samples[:, pc_x], samples[:, pc_y],
                   color="cornflowerblue", s=8, alpha=0.2, zorder=2)
    
    # 1-sigma and 2-sigma ellipses
    from matplotlib.patches import Ellipse
    for n_sigma, alpha in [(1, 0.3), (2, 0.15)]:
        w = 2 * n_sigma * np.sqrt(pred_cov[pc_x])
        h = 2 * n_sigma * np.sqrt(pred_cov[pc_y])
        ellipse = Ellipse(xy=(pred_mean[pc_x], pred_mean[pc_y]),
                          width=w, height=h,
                          facecolor="cornflowerblue", alpha=alpha,
                          edgecolor="blue", lw=1, zorder=2)
        ax.add_patch(ellipse)
    
    # Predicted mean (red star)
    ax.scatter(pred_mean[pc_x], pred_mean[pc_y], color="red", s=100,
               zorder=5, marker="*", edgecolors="k", linewidths=0.5)
    
    # Target (gold diamond)
    ax.scatter(tgt[pc_x], tgt[pc_y], color="gold", s=120, zorder=6,
               marker="D", edgecolors="k", linewidths=1.5)
    
    dt = eval_batch.horizon_dt[idx].item()
    R_e = pred_result.rate[idx].item() if pred_result.rate is not None else 0
    ax.set_title(f"dt={dt:.0f}s  R_e={R_e:.3f}", fontsize=9)
    ax.set_xlabel("PC1", fontsize=8)
    ax.set_ylabel("PC2", fontsize=8)

plt.suptitle("Model predictions: red star=predicted mean, gold diamond=target\n"
             "blue ellipses=1σ/2σ, light dots=forward samples", fontsize=12)
plt.tight_layout()
plt.show()

---
## 7. Compare with baselines

In [ ]:
from dev.dynamo.eval import run_evaluation
from dev.dynamo.eval.predictions import PersistencePredictor
from dev.dynamo.models import SimpleKernelPredictor
from collections import OrderedDict

# Re-create eval_frag with more samples
eval_frag._epoch_length = 1500

comparisons = OrderedDict([
    ("Persistence", PersistencePredictor(noise_scale=0.1)),
    ("Kernel",      SimpleKernelPredictor(dataset, n_samples=50, exclude_self=True)),
    ("Phi0 Model",  model),
])

print(f"{'Model':<18s}  {'NLL':>10s}  {'MSE':>10s}  {'RMSE':>10s}  {'Cal':>8s}")
print("-" * 62)

for name, predictor in comparisons.items():
    if hasattr(predictor, 'eval'):
        predictor.eval()
    result = run_evaluation(predictor, eval_frag, n_batches=30, batch_size=32)
    print(f"{name:<18s}  {result.metrics['nll']:>10.4f}  "
          f"{result.metrics['mse']:>10.6f}  {result.metrics['rmse']:>10.6f}  "
          f"{result.calibration:>8.3f}")

---
## 8. Save/load checkpoint (optional)

In [ ]:
# Save
ckpt_path = Path("../checkpoints/notebook_demo.pt")
ckpt_path.parent.mkdir(parents=True, exist_ok=True)

torch.save({
    "model_state_dict": model.state_dict(),
    "input_dim": dataset.n_components,
    "config": {
        "hidden_dim": HIDDEN_DIM,
        "n_hidden": N_HIDDEN,
        "activation": ACTIVATION,
        "alpha_0": ALPHA_0,
    },
    "history": history,
    "experiment_ids": EXPERIMENT_IDS,
}, ckpt_path)
print(f"Saved checkpoint to {ckpt_path}")

# Load (demonstration)
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
loaded_model = Phi0OnlyModel(
    input_dim=ckpt["input_dim"],
    hidden_dim=ckpt["config"]["hidden_dim"],
    n_hidden=ckpt["config"]["n_hidden"],
    activation=ckpt["config"]["activation"],
)
loaded_model.load_state_dict(ckpt["model_state_dict"])
loaded_model.eval()
print(f"Loaded model: beta={loaded_model.beta.item():.4f}, D={loaded_model.D.item():.6f}")